# 01 — Primeros pasos con Pyvium

**Pyvium** es un wrapper de Python alrededor de la "Software Development Driver DLL" de IviumSoft que permite controlar dispositivos de electroquímica Ivium (CompactStat, etc.) de forma programática.

### Requisitos

- **Solo Windows** — la DLL subyacente es específica de Windows
- **IviumSoft debe estar instalado** (versión 4.1239 o compatible)
- **IviumSoft debe estar en ejecución** antes de llamar a la mayoría de las funciones de Pyvium
- Python 3.11 o 3.12

---

## Arquitectura de la biblioteca

Pyvium expone tres capas independientes:

```
┌─────────────────────────────────────────────────────────┐
│  from pyvium import Pyvium   ← usar esto para experimentos │
│  Wrapper de alto nivel — nombres Pythónicos, lanza excepciones │
├─────────────────────────────────────────────────────────┤
│  from pyvium import Core     ← llamadas directas a la DLL      │
│  Wrappers directos de DLL — devuelve tuplas (result_code, value) │
├─────────────────────────────────────────────────────────┤
│  from pyvium import Tools    ← no requiere hardware       │
│  Parseo de archivos IDF, exportación a CSV — funciona sin conexión │
└─────────────────────────────────────────────────────────┘
```

Esta serie de notebooks cubre la capa **Pyvium** (la API recomendada). Consulta el notebook `07_data_processing.ipynb` para la capa Tools.

## 1. Importar la biblioteca

In [ ]:
from pyvium import Pyvium

# Importar tipos de error para manejo explícito de excepciones
from pyvium.errors import (
    DriverNotOpenError,
    IviumSoftNotRunningError,
    DeviceNotConnectedToIviumSoftError,
    NoDeviceDetectedError,
    DeviceBusyError,
)

print("pyvium imported successfully")

## 2. Verificación de versión

Antes de conectar cualquier hardware, verifica que la DLL incluida con pyvium sea compatible con la versión instalada de IviumSoft. Una discrepancia de versiones puede causar fallos silenciosos o bloqueos.

> **Nota:** `open_driver()` debe llamarse antes que cualquier otra función. Establece el canal de comunicación con IviumSoft.

In [ ]:
Pyvium.open_driver()

print(f"DLL version       : {Pyvium.get_dll_version()}")
print(f"DLL version string: {Pyvium.get_dll_version_string()}")
print(f"IviumSoft version : {Pyvium.get_iviumsoft_version()}")
print(f"Required DLL ver  : {Pyvium.get_version_host()}")
print(f"Version match     : {Pyvium.check_dll_version()}")

## 3. Ciclo de vida del driver

El driver es el canal de comunicación entre Python e IviumSoft. Ábrelo siempre antes de usarlo y ciérralo cuando hayas terminado.

| Paso | Método | Efecto |
|------|--------|--------|
| 1 | `open_driver()` | Conecta Python al proceso IviumSoft en ejecución |
| 2 | *(trabajar)* | Medir, configurar, ejecutar métodos... |
| 3 | `close_driver()` | Libera la conexión — IviumSoft sigue ejecutándose |

Llamar a `open_driver()` cuando el driver ya está abierto lo cierra y reabre automáticamente (es seguro llamarlo múltiples veces).

In [ ]:
# Patrón recomendado: usar try/finally para garantizar que el driver se cierre
try:
    Pyvium.open_driver()
    print("Driver open — ready to communicate with IviumSoft")

    # ... tu código de medición va aquí ...

finally:
    Pyvium.close_driver()
    print("Driver closed")

## 4. Manejo de errores

La capa Pyvium traduce los códigos de retorno brutos de la DLL en excepciones tipadas de Python. Cada excepción corresponde a un estado específico del dispositivo o resultado de un comando:

| Excepción | Código DLL | Significado | Solución |
|-----------|----------|---------|-----|
| `DriverNotOpenError` | — | `open_driver()` no fue llamado | Llamar `open_driver()` primero |
| `IviumSoftNotRunningError` | — | El proceso IviumSoft no está en ejecución | Iniciar IviumSoft |
| `DeviceNotConnectedToIviumSoftError` | — | Dispositivo encontrado pero no conectado en IviumSoft | Hacer clic en Conectar en IviumSoft, o llamar `connect_device()` |
| `NoDeviceDetectedError` | `-1` | No se detectó ningún dispositivo (también se lanza por código de resultado -1 del setter) | Verificar cable USB e instalación del driver |
| `DeviceBusyError` | — | El dispositivo ya está ejecutando una medición | Esperar o llamar `abort_method()` |
| `CellOffError` | — | Se intentó medir con la celda apagada | Llamar `set_cell_on()` primero |
| `IllegalCommandError` | `1` | Comando no válido para este dispositivo o su modo actual | Verificar el modo de conexión y las capacidades del dispositivo |
| `InvalidStateError` | `3` | El comando es válido pero el estado del dispositivo lo impide | Esperar a que termine cualquier medición en curso |
| `ValueError` (integrado) | `2` | Argumento fuera de rango — el firmware rechazó el valor | Verificar el rango válido para el parámetro |

In [ ]:
# Ejemplo: manejo elegante de errores
try:
    Pyvium.open_driver()
    Pyvium.connect_device()
    print("Device connected")

    # ... código de medición ...

except IviumSoftNotRunningError:
    print("ERROR: IviumSoft is not running. Please start it and try again.")

except DeviceNotConnectedToIviumSoftError:
    print("ERROR: No device connected. Check the USB cable and IviumSoft.")

except NoDeviceDetectedError:
    print("ERROR: Device not detected. Check USB drivers.")

except DeviceBusyError:
    print("ERROR: Device is busy. Wait for the current measurement to finish.")

finally:
    try:
        Pyvium.close_driver()
    except DriverNotOpenError:
        pass  # open_driver() pudo haber fallado antes de que pudiéramos cerrar

## 5. Verificar si IviumSoft está en ejecución

Antes de entrar en mediciones, puedes sondear el estado del sistema sin lanzar excepciones.

In [ ]:
Pyvium.open_driver()

running = Pyvium.is_iviumsoft_running()
print(f"IviumSoft running: {running}")

if running:
    status_code, status_label = Pyvium.get_device_status()
    print(f"Device status: {status_code} → '{status_label}'")
    # Significados de status_code:
    #  -1 : sin IviumSoft
    #   0 : no conectado
    #   1 : disponible, inactivo
    #   2 : disponible, ocupado (ejecutando un método)
    #   3 : no hay dispositivo disponible

Pyvium.close_driver()

## 6. Ejemplo mínimo funcional

El script completo más simple: abrir el driver, confirmar que IviumSoft está en ejecución, luego cerrar.

In [ ]:
from pyvium import Pyvium
from pyvium.errors import IviumSoftNotRunningError

try:
    Pyvium.open_driver()

    if not Pyvium.is_iviumsoft_running():
        raise IviumSoftNotRunningError()

    status_code, status_label = Pyvium.get_device_status()
    print(f"Status: {status_label}")
    print(f"DLL OK: {Pyvium.check_dll_version()}")
    print("Setup verified — ready to proceed")

finally:
    try:
        Pyvium.close_driver()
    except Exception:
        pass

---

## Resumen

| Concepto | Puntos clave |
|---------|------------|
| Importar | `from pyvium import Pyvium` |
| Siempre primero | `Pyvium.open_driver()` |
| Siempre último | `Pyvium.close_driver()` (usar `try/finally`) |
| Verificar versión | `Pyvium.check_dll_version()` debe devolver `True` |
| Estado del dispositivo | `Pyvium.get_device_status()` → `(code, label)` |
| Tipos de error | 8 excepciones tipadas — 6 para estado del dispositivo/conexión, 2 nuevas para códigos de resultado del setter |

## Códigos de retorno de DLL (capa Core)

Si alguna vez usas la capa `Core` directamente, todas las funciones DLL devuelven un código de estado entero:

| Código | Significado |
|------|---------| 
| `0` | Éxito |
| `-1` | Sin dispositivo / IviumSoft no está en ejecución |
| `1` | Comando ilegal |
| `2` | Argumento fuera de rango |
| `3` | Estado inválido |

La capa `Pyvium` traduce estos en excepciones tipadas automáticamente — solo ves los códigos brutos si usas `Core` directamente.

## Próximos pasos

- **`02_device_and_instance_management.ipynb`** — Seleccionar dispositivos, gestionar múltiples instancias de IviumSoft, conectar por número de serie
- **`03_direct_mode_basics.ipynb`** — Controlar potencial/corriente directamente sin un archivo de método
- **`07_data_processing.ipynb`** — Parsear archivos IDF sin conexión (no se requiere hardware)